# 📘 Notebook 01: Tables and SQL

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module A: Data at Rest · Notebook 1 of 4 in this module · (1 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what goes wrong when all your data is kept in **one big table**
- Describe the **relational model**: tables, rows, columns, primary keys and foreign keys
- Create a database with **SQLite** from Python and fill it with data
- Write **SQL** queries that select, filter, sort and limit
- Explain why SQL is called **declarative**, and what that buys you
- Change data safely with **transactions**, and pass values to a query without opening a security hole

> **Prerequisites:** Python basics: loops, functions, lists and dictionaries.
>
> 🔭 **Why this notebook matters:** nearly every organisation in the world keeps its most valuable data in tables, and talks to those tables in one language, SQL, which is about fifty years old and shows no sign of retiring. Everything later in this course, from data warehouses to Spark, either builds on the ideas of this notebook or defines itself against them. We begin where the data begins: with one table, and the moment one table stops being enough.

> 📦 **Libraries used in this notebook.** None that need installing. **SQLite** is a complete relational database engine contained in a single small library, and it is built into Python as the module `sqlite3`. It is the most widely deployed database in the world: there is a copy in every phone and every web browser. It needs no server, no password and no setup, which makes it ideal for learning.

In [ ]:
import random
import datetime
import sqlite3

## 1. One big table, and why it fails

### Intuition first
Imagine you open a small online bookshop and decide to keep your records in a single spreadsheet, one row per sale. In Python, that is a list of dictionaries.

In [ ]:
sales = [
    {"order": 1, "customer": "Anna Markou",   "city": "Athens",       "book": "The Salt Road",  "price": 12.50, "quantity": 1},
    {"order": 1, "customer": "Anna Markou",   "city": "Athens",       "book": "A Winter Atlas", "price": 18.00, "quantity": 2},
    {"order": 2, "customer": "Nikos Pappas",  "city": "Patras",       "book": "The Salt Road",  "price": 12.50, "quantity": 1},
    {"order": 3, "customer": "Anna Markou",   "city": "Athens",       "book": "Paper Lanterns", "price": 9.90,  "quantity": 1},
    {"order": 4, "customer": "Eleni Vlachou", "city": "Thessaloniki", "book": "A Winter Atlas", "price": 18.00, "quantity": 1},
]

for row in sales:
    if row["city"] == "Athens":
        print(row["order"], row["customer"], row["book"])

It works. Now three ordinary things happen in the life of the shop.

**Anna moves to Patras.** Her city is written in three rows. Miss one when updating, and the data now says two different things about where she lives.

**You add a new book that nobody has bought yet.** There is no row to put it in, because every row is a sale.

**You delete Eleni's only order.** Her name and city vanish with it, although she is still your customer.

In [ ]:
for row in sales:
    if row["customer"] == "Anna Markou" and row["order"] == 1:      # an update that misses a row
        row["city"] = "Patras"

cities = set(row["city"] for row in sales if row["customer"] == "Anna Markou")
print("Cities recorded for Anna:", cities)

The table now contradicts itself. These three problems have names: the **update**, **insertion** and **deletion anomalies**. They all come from one cause. A single table is being used to record several different kinds of thing (customers, books, orders), so the same fact is stored many times.

The cure is to store **each fact once**.

## 2. The relational model

### Formal definition
In the **relational model**, proposed by Edgar Codd in 1970:

- Data is kept in **tables** (formally, *relations*). Each table describes one kind of thing.
- Each **row** is one thing of that kind. Each **column** is one property, with a fixed type.
- Each table has a **primary key**: a column whose value is different in every row, and which identifies the row.
- A table refers to a row of another table by storing its key. Such a column is a **foreign key**.

For the bookshop this gives four tables.

```
 customers                products                 orders                    order_items
 ---------                --------                 ------                    -----------
 customer_id  (key)       product_id  (key)        order_id     (key)        item_id     (key)
 name                     title                    customer_id  ──► customers order_id    ──► orders
 city                     category                 order_date                product_id  ──► products
 birth_year               price                                              quantity
```

Anna's city is now written in exactly one place. A book can exist without being sold, and a customer without an order. The splitting of one table into several, so that each fact is stored once, is called **normalisation**.

### Creating the tables
We talk to the database in **SQL**, the Structured Query Language. The statement `CREATE TABLE` defines a table, its columns and their types.

In [ ]:
db = sqlite3.connect(":memory:")            # a database that lives in memory, for this notebook only
db.execute("PRAGMA foreign_keys = ON")      # ask SQLite to enforce foreign keys

db.executescript("""
CREATE TABLE customers (
    customer_id  INTEGER PRIMARY KEY,
    name         TEXT NOT NULL,
    city         TEXT,
    birth_year   INTEGER
);
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    title        TEXT NOT NULL,
    category     TEXT,
    price        REAL
);
CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date   TEXT
);
CREATE TABLE order_items (
    item_id      INTEGER PRIMARY KEY,
    order_id     INTEGER NOT NULL REFERENCES orders(order_id),
    product_id   INTEGER NOT NULL REFERENCES products(product_id),
    quantity     INTEGER
);
""")
print("Tables created.")

### Some data to work with
Five rows are too few to learn from. The function below invents a bookshop of any size: customers in Greek cities, books in eight categories, and two years of orders. Every notebook of this course starts from the same function, so the data will become familiar.

In [ ]:
def make_shop(customers=1000, products=200, orders=10000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]

    customer_rows = [(i, f"Customer {i}", rng.choice(cities), 1950 + rng.randint(0, 55)) for i in range(1, customers + 1)]
    product_rows = [(i, f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(1, products + 1)]

    order_rows, item_rows = [], []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        order_rows.append((order_id, rng.randint(1, customers), day.isoformat()))
        for line in range(rng.randint(1, 4)):
            item_rows.append((len(item_rows) + 1, order_id, rng.randint(1, products), rng.randint(1, 3)))
    return customer_rows, product_rows, order_rows, item_rows

customer_rows, product_rows, order_rows, item_rows = make_shop()

db.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customer_rows)
db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", product_rows)
db.executemany("INSERT INTO orders VALUES (?, ?, ?)", order_rows)
db.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", item_rows)
db.commit()

for table in ["customers", "products", "orders", "order_items"]:
    print(f"{table:<12} {db.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:>7,} rows")

## 3. Asking questions: SELECT

### Intuition first
To get data out, you **describe the rows you want**, and the database works out how to find them. The basic sentence of SQL reads almost like English:

```sql
SELECT   which columns
FROM     which table
WHERE    which rows
ORDER BY in what order
LIMIT    how many
```

A small helper will run a query and print the result as a table.

In [ ]:
def show(sql, parameters=(), limit=8):
    cursor = db.execute(sql, parameters)
    columns = [description[0] for description in cursor.description]
    rows = cursor.fetchall()
    print(" | ".join(f"{column:<14}" for column in columns))
    print("-" * (17 * len(columns) - 3))
    for row in rows[:limit]:
        print(" | ".join(f"{str(value):<14}" for value in row))
    if len(rows) > limit:
        print(f"... {len(rows):,} rows in total")
    print()

show("SELECT * FROM customers", limit=5)

`*` means "all columns". Usually you want only some of them, and only some of the rows.

In [ ]:
show("""
    SELECT name, city, birth_year
    FROM customers
    WHERE city = 'Patras' AND birth_year >= 2000
    ORDER BY birth_year DESC, name
""", limit=6)

Read the query aloud: *the name, city and birth year of the customers in Patras born in 2000 or later, youngest first*. A few more tools cover most everyday questions.

| Clause | Meaning | Example |
|---|---|---|
| `AND`, `OR`, `NOT` | combine conditions | `city = 'Volos' OR city = 'Chania'` |
| `IN (...)` | one of a list | `city IN ('Volos', 'Chania')` |
| `BETWEEN a AND b` | a range, ends included | `price BETWEEN 10 AND 20` |
| `LIKE` | a text pattern: `%` is any text | `title LIKE 'Book 1%'` |
| `DISTINCT` | remove repeated rows | `SELECT DISTINCT city` |
| `LIMIT n` | at most n rows | `LIMIT 5` |

In [ ]:
show("SELECT DISTINCT category FROM products ORDER BY category", limit=10)

show("""
    SELECT title, category, price
    FROM products
    WHERE category IN ('Science', 'Computing') AND price BETWEEN 50 AND 60
    ORDER BY price DESC
    LIMIT 5
""")

### Declarative, not procedural
Compare the query above with the loop at the start of this notebook. The loop said **how** to find the rows: look at each one in turn and test it. The query says only **what** is wanted. This style is called **declarative**.

It looks like a small difference, and it is the central idea of databases. Because you have not said how, the database is free to choose: to read the whole table, to use a shortcut it has prepared, or to split the work across many machines. The same query text can run on five rows in memory or on five billion rows in a cluster. Notebook 03 shows the database making such choices, and by Notebook 10 the same SQL will be running on Spark.

## 4. The missing value: NULL

A customer may not have told us their city. The database records this with a special marker, `NULL`, which means "unknown". It is not zero and it is not an empty string, and it behaves in a way that surprises everyone once.

In [ ]:
db.execute("INSERT INTO customers VALUES (5001, 'Customer 5001', NULL, 1990)")
db.commit()

print("city = NULL  :", db.execute("SELECT COUNT(*) FROM customers WHERE city = NULL").fetchone()[0])
print("city IS NULL :", db.execute("SELECT COUNT(*) FROM customers WHERE city IS NULL").fetchone()[0])
print("city != 'Athens'           :", db.execute("SELECT COUNT(*) FROM customers WHERE city != 'Athens'").fetchone()[0])
print("everyone minus the Athenians:", db.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
      - db.execute("SELECT COUNT(*) FROM customers WHERE city = 'Athens'").fetchone()[0])

`city = NULL` finds nothing, although such a customer exists. Is an unknown value equal to an unknown value? The database's answer is "unknown", and a row is returned only when the condition is definitely true. The correct test is `IS NULL`.

The last two lines show the same trap from another side. The customers "not in Athens" and "everyone minus the Athenians" differ by one: the customer whose city is unknown is in neither group.

> ⚠️ **Common pitfalls**
>
> - Testing for a missing value with `= NULL`. It never matches. Use `IS NULL` or `IS NOT NULL`.
> - Forgetting that `WHERE column != 'x'` silently drops the rows where the column is NULL.
> - Using `SELECT *` in real programs. It fetches columns you do not need, and your code breaks when someone adds a column. Name the columns you want.
> - Relying on the order of rows without `ORDER BY`. A table has no natural order, and the database may return rows in any order it finds convenient.

## 5. Changing data, safely

Three statements change the contents of a table: `INSERT` adds rows, `UPDATE` changes them and `DELETE` removes them. `UPDATE` and `DELETE` take a `WHERE` clause, and they apply to **every** row that matches.

Anna moving to another city is now a change in exactly one place.

In [ ]:
show("SELECT customer_id, name, city FROM customers WHERE customer_id = 7")

db.execute("UPDATE customers SET city = 'Patras' WHERE customer_id = 7")
db.commit()

show("SELECT customer_id, name, city FROM customers WHERE customer_id = 7")

### Transactions
What if a change consists of several steps, and something fails halfway? A **transaction** is a group of changes that the database treats as one: either all of them happen, or none do. Until you `commit`, you can undo everything with `rollback`.

Watch what happens when someone forgets the `WHERE`.

In [ ]:
db.execute("UPDATE products SET price = 0")                      # oops: no WHERE, so every book is now free
print("Books costing nothing:", db.execute("SELECT COUNT(*) FROM products WHERE price = 0").fetchone()[0])

db.rollback()                                                    # nothing was committed, so undo it
print("After rollback:       ", db.execute("SELECT COUNT(*) FROM products WHERE price = 0").fetchone()[0])

### The database defends its own rules
We declared keys when we created the tables. The database now refuses any change that would break them.

In [ ]:
attempts = {
    "a second customer with id 7":      "INSERT INTO customers VALUES (7, 'Impostor', 'Athens', 1980)",
    "an order for a customer who does not exist": "INSERT INTO orders VALUES (999999, 424242, '2024-06-01')",
    "a customer without a name":        "INSERT INTO customers VALUES (6001, NULL, 'Athens', 1980)",
}
for description, statement in attempts.items():
    try:
        db.execute(statement)
        print(f"accepted: {description}")
    except sqlite3.IntegrityError as problem:
        print(f"REFUSED:  {description}\n          ({problem})")
db.rollback()

These rules are called **constraints**. A loop over a list of dictionaries would have accepted all three without complaint. Guarding the consistency of the data, whoever is writing to it and whatever program they use, is half of what a database is for.

## 6. Passing values into a query

In a real program, parts of a query come from outside: a city typed into a search box, say. The tempting way is to paste the value into the SQL text.

In [ ]:
def customers_in_unsafe(city):
    sql = f"SELECT COUNT(*) FROM customers WHERE city = '{city}'"       # DO NOT do this
    return db.execute(sql).fetchone()[0]

print(customers_in_unsafe("Volos"))
print(customers_in_unsafe("nowhere' OR '1' = '1"))

The second call asked for a city called *nowhere* and received a count of **every** customer. The text typed by the user contained a quotation mark, which ended the string early, and the rest of what they typed became part of the query: `... WHERE city = 'nowhere' OR '1' = '1'`. With a different input the same trick can read other tables or delete data. This is **SQL injection**, and for more than twenty years it has been one of the commonest causes of stolen data.

The defence is simple and complete. Never build SQL by gluing strings together. Put a `?` where each value goes, and pass the values separately. The database then treats them strictly as data.

In [ ]:
def customers_in(city):
    return db.execute("SELECT COUNT(*) FROM customers WHERE city = ?", (city,)).fetchone()[0]

print(customers_in("Volos"))
print(customers_in("nowhere' OR '1' = '1"))

Now the strange input is just the name of a city that has no customers.

## 7. Where we stand

The question of this course is: **what changes when the data no longer fits?** In this notebook it stopped fitting in **one table**.

| The problem | The answer |
|---|---|
| The same fact stored in many rows | several tables, each fact stored once |
| Rows that must refer to each other | primary keys and foreign keys |
| Loops written by hand for every question | a declarative language, SQL |
| Data left inconsistent by a mistake or a crash | constraints and transactions |

The price of splitting the data is that the answer to an ordinary question, such as *"which books did Anna buy?"*, now lies across four tables. Putting the pieces back together is the subject of the next notebook.

---
## ✏️ Exercises

### Exercise 1 (First queries)
Write queries for: (a) the titles and prices of the five most expensive books, (b) the number of customers in Heraklion, (c) all the different cities that customers live in, in alphabetical order.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show("SELECT title, price FROM products ORDER BY price DESC LIMIT 5")
show("SELECT COUNT(*) AS customers FROM customers WHERE city = 'Heraklion'")
show("SELECT DISTINCT city FROM customers ORDER BY city", limit=20)
```

*The last result includes a row for the customer whose city is unknown. `DISTINCT` treats all the NULLs as one value. `AS` gives a column of the result a readable name.*
</details>

### Exercise 2 (Filters)
Find the customers born between 1990 and 1995 who live in Athens or Thessaloniki. Show the name, city and birth year, oldest first, and only the first ten.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show("""
    SELECT name, city, birth_year
    FROM customers
    WHERE birth_year BETWEEN 1990 AND 1995
      AND city IN ('Athens', 'Thessaloniki')
    ORDER BY birth_year, name
    LIMIT 10
""", limit=10)
```

*Had you written `city = 'Athens' OR city = 'Thessaloniki'` without brackets after the `AND`, the result would be wrong: `AND` binds more tightly than `OR`. `IN` avoids the problem.*
</details>

### Exercise 3 (The same question, two ways)
Count the orders placed in December 2023, first with a Python loop over `order_rows` (each row is `(order_id, customer_id, order_date)`), and then with one SQL query. Dates are stored as text in the form `2023-12-31`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
count = 0
for order_id, customer_id, order_date in order_rows:
    if order_date.startswith("2023-12"):
        count += 1
print("Loop:", count)

print("SQL: ", db.execute("SELECT COUNT(*) FROM orders WHERE order_date LIKE '2023-12%'").fetchone()[0])
```

*Dates written as year-month-day sort and compare correctly as plain text, which is why that format is the standard one for storing them. `order_date BETWEEN '2023-12-01' AND '2023-12-31'` works as well.*
</details>

### Exercise 4 (A safe change)
The shop decides to raise the price of every book in the `Cooking` category by 10%. Write the `UPDATE`, check with a `SELECT` that only cooking books changed, and then `rollback` so that the notebook's data stays as it was.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
before = db.execute("SELECT ROUND(AVG(price), 2) FROM products WHERE category = 'Cooking'").fetchone()[0]
others = db.execute("SELECT ROUND(AVG(price), 2) FROM products WHERE category != 'Cooking'").fetchone()[0]

db.execute("UPDATE products SET price = ROUND(price * 1.10, 2) WHERE category = 'Cooking'")

print("Cooking:", before, "->", db.execute("SELECT ROUND(AVG(price), 2) FROM products WHERE category = 'Cooking'").fetchone()[0])
print("Others: ", others, "->", db.execute("SELECT ROUND(AVG(price), 2) FROM products WHERE category != 'Cooking'").fetchone()[0])

db.rollback()
print("After rollback:", db.execute("SELECT ROUND(AVG(price), 2) FROM products WHERE category = 'Cooking'").fetchone()[0])
```

*Checking a change before committing it, and being able to withdraw it, is the everyday use of a transaction. Professionals run the `SELECT` with the same `WHERE` first, to see which rows the `UPDATE` will touch.*
</details>

### Exercise 5 (Break it, then fix it)
The function below is open to SQL injection. First find an input for `title_start` that makes it return **every** book. Then rewrite the function safely with a `?` placeholder and show that your input no longer works.

In [ ]:
def find_books_unsafe(title_start):
    return db.execute(f"SELECT COUNT(*) FROM products WHERE title LIKE '{title_start}%'").fetchone()[0]

print(find_books_unsafe("Book 19"))
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
attack = "zzz' OR title LIKE '"
print("Unsafe:", find_books_unsafe(attack))

def find_books(title_start):
    return db.execute("SELECT COUNT(*) FROM products WHERE title LIKE ?", (title_start + "%",)).fetchone()[0]

print("Safe:  ", find_books("Book 19"), find_books(attack))
```

*The attack closes the quotation mark and adds a second condition that matches everything. With a placeholder, the whole input, quotation mark included, is treated as a title to search for, and no book has such a title. Note that the `%` is added to the value, not written into the SQL.*
</details>

### Exercise 6 (Design a table, a little harder)
The shop wants to record **reviews**: a customer gives a book a rating from 1 to 5, on a date, with an optional comment. Write a `CREATE TABLE` statement with a primary key, the two foreign keys, and a rule that the rating must be between 1 and 5 (look up the `CHECK` constraint). Insert one valid review, and show that a rating of 7 and a review of a book that does not exist are both refused.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
db.execute("""
    CREATE TABLE reviews (
        review_id    INTEGER PRIMARY KEY,
        customer_id  INTEGER NOT NULL REFERENCES customers(customer_id),
        product_id   INTEGER NOT NULL REFERENCES products(product_id),
        rating       INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
        review_date  TEXT,
        comment      TEXT
    )
""")
db.execute("INSERT INTO reviews VALUES (1, 7, 12, 5, '2024-03-01', 'Wonderful')")

for description, statement in [("a rating of 7", "INSERT INTO reviews VALUES (2, 7, 12, 7, '2024-03-02', NULL)"),
                               ("a book that does not exist", "INSERT INTO reviews VALUES (3, 7, 99999, 4, '2024-03-02', NULL)")]:
    try:
        db.execute(statement)
        print("accepted:", description)
    except sqlite3.IntegrityError as problem:
        print(f"REFUSED:  {description} ({problem})")
db.commit()
```

*A review is a fact about a pair, one customer and one book, so it gets a table of its own with a foreign key to each. The comment may be NULL. The rating may not, and the database itself now guarantees that it is sensible, whatever program writes to the table.*
</details>

## 🔑 Key takeaways

- One big table stores the same fact many times, which leads to **update, insertion and deletion anomalies**.
- The **relational model** keeps one kind of thing per table and each fact once. Rows are identified by a **primary key** and linked by **foreign keys**.
- **SQL** is **declarative**: you describe the rows you want, and the database decides how to find them.
- `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT` covers most everyday questions.
- `NULL` means unknown. Test for it with `IS NULL`, never with `=`.
- **Constraints** and **transactions** keep data consistent: a change either happens completely or not at all.
- Never paste values into SQL text. Use **placeholders**, or you invite SQL injection.

---
**Next:** *Notebook 02: Joins and Aggregation*, where we put the four tables back together, and learn to summarise thousands of rows in one line.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*